# Attention Pattern Evolution

Track how attention patterns change across layers: stability, focus shifts,
head agreement, entropy evolution, and summary.

## Why This Matters

Attention pattern evolution reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_pattern_evolution import (
    attention_stability_across_layers, attention_focus_evolution,
    head_agreement_evolution, attention_entropy_evolution,
    attention_evolution_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Attention Stability

How stable are attention patterns for the same head across layers?

In [ ]:
for head in range(4):
    result = attention_stability_across_layers(model, tokens, head=head)
    flag = ' [STABLE]' if result['is_stable'] else ''
    sims = ', '.join(f'{p["cosine_similarity"]:.3f}' for p in result['per_pair'])
    print(f"  Head {head}: mean={result['mean_stability']:.4f} [{sims}]{flag}")

## Focus Evolution

Where does the last position focus attention at each layer?

In [ ]:
result = attention_focus_evolution(model, tokens, position=-1)
print(f"Focus shifts: {result['focus_shifts']}\n")
for p in result['per_layer']:
    focus = ', '.join(f'pos{f["position"]}({f["attention"]:.3f})' for f in p['top_focus'])
    print(f"  Layer {p['layer']}: H={p['entropy']:.3f} | {focus}")

## Head Agreement

Do heads agree on where to attend?

In [ ]:
result = head_agreement_evolution(model, tokens)
print(f"Agreement trend: {result['agreement_trend']}\n")
for p in result['per_layer']:
    bar = '█' * int(max(0, p['mean_head_agreement']) * 30)
    flag = ' [AGREE]' if p['heads_agree'] else ''
    print(f"  Layer {p['layer']}: agreement={p['mean_head_agreement']:.4f} {bar}{flag}")

## Entropy Evolution

Does attention sharpen or broaden through the network?

In [ ]:
result = attention_entropy_evolution(model, tokens)
print(f"Sharpens: {result['sharpens']}")
print(f"Entropy range: {result['entropy_range']:.4f}\n")
for p in result['per_layer']:
    heads = ', '.join(f'H{i}:{e:.2f}' for i, e in enumerate(p['per_head_entropy']))
    print(f"  Layer {p['layer']}: mean={p['mean_entropy']:.3f} | {heads}")

## Evolution Summary

Cross-layer overview of attention pattern evolution.

In [ ]:
result = attention_evolution_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: H={p['mean_entropy']:.3f}, "
          f"stability={p['stability_from_prev']:.4f}, "
          f"self_attn={p['self_attention']:.4f}")